In [ ]:
from pathlib import Path
import os
import sys
import json

REPO_ROOT = (
    Path.cwd() if (Path.cwd() / "diffusion_project").is_dir() else Path.cwd().parent
)
sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)
# All experiment and display settings. Edit this cell, then Restart & Run All.
from pathlib import Path

TOPDIR = Path(os.environ.get("EXPERIMENT_ROOT", "./results"))
DATASET = "celeba32"
MODEL_TYPE = "dit"  # 'dit' or 'unet'
RUN_SUFFIX = "_b"  # '', '_b', '_c'
MODEL_RUN_SUFFIXES = {"dit": RUN_SUFFIX, "unet": RUN_SUFFIX}
CHECKPOINT_NAME_TEMPLATE = (
    "{kind}_10k_s{subset}{suffix}"  # naming template for 10000-example subsets
)
SUBSET_SIZE = 10000
ANALYSIS_SUBSET = 1  # 1 or 2, for trajectory/single-epoch plots
DEVICE = "auto"  # 'auto', 'cuda', or 'cpu'

# None derives paths from TOPDIR and DATASET.
CKPT_ROOT = None
SAVED_TENSORS_DIR = None
SEEDS_FILE = None
REFERENCE_FILES = {1: None, 2: None}
# Optional explicit overrides, e.g. {('dit', 1): Path('/.../dit_10k_s1_b')}.
CHECKPOINT_DIR_OVERRIDES = {}

NUM_SAMPLES = 100  # uses the common available prefix if shorter
SAMPLING_BATCH_SIZE = 8  # controls GPU memory, not training batch size
SAMPLE_STEPS = 100
AVG_K = 5  # weight averaging over preceding epoch window
CACHE_SAMPLES = True  # CPU cache; reset by rerunning setup

# Section switches. Disable anything you do not want to compute/display.
RUN_TRAJECTORY = False
RUN_SINGLE_EPOCH = True
RUN_SUBSET_PAIR = True
RUN_EARLY_LATE = True
RUN_CROSS_MODEL = True
TRAJECTORY_EPOCHS = list(
    range(10, 401, 10)
)  # missing target epochs are reported/skipped
SINGLE_EPOCH = 30
PAIR_EPOCHS = {1: 15, 2: 20}
EARLY_LATE_EPOCHS = (30, 390)
CROSS_MODEL_EPOCHS = {"dit": 15, "unet": 10}
# For non-trajectory selections, use None for the latest checkpoint.


# Figure appearance and sample selection.
PLOT_INDICES = list(range(5))  # indices of the samples to display
FONT_FAMILY = "serif"
SERIF_FONTS = ["Times New Roman", "DejaVu Serif"]
IMAGE_CMAP = "Greys_r"
ERROR_CMAP = "viridis"
SIGNED_ERROR_CMAP = "seismic"
INDIVIDUAL_FIGSIZE = (6.4, 4.8)
PAIR_FIGSIZE = (8, 8)
EARLY_LATE_FIGSIZE = (24, 16)
CROSS_FIGSIZE = (12, 8)
GRID_TITLE_SIZE = 35
GRID_TITLE_PAD = 15
# Cross-model grid variants: titled and untitled.
CROSS_GRID_VARIANTS = ("titled", "plain")
REFERENCE_LABEL = "Analytical"
PLAIN_CROSS_TITLES = False
TITLED_CROSS_RECT = (0.15, 0, 1, 1)

SAVE_FIGURES = False
FIGURE_DIR = TOPDIR / "figures" / DATASET


# Checkpoint comparisons

Display individual samples, two-subset comparisons, early/late training pairs, and
analytical/UNet/DiT grids. Select the desired sections and checkpoints in the first cell.
Cross-model defaults are UNet epoch 10 and DiT epoch 15, with a trailing window of
up to five checkpoints. Figure export is optional.


In [ ]:
import os
import math
import warnings
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, Subset
import sys

if MODEL_TYPE not in ("dit", "unet") or ANALYSIS_SUBSET not in (1, 2):
    raise ValueError("Choose MODEL_TYPE dit/unet and ANALYSIS_SUBSET 1/2")
if min(SUBSET_SIZE, NUM_SAMPLES, SAMPLING_BATCH_SIZE, AVG_K) < 1:
    raise ValueError("Sizes and averaging window must be positive")
if any(kind not in ("titled", "plain") for kind in CROSS_GRID_VARIANTS):
    raise ValueError("CROSS_GRID_VARIANTS accepts titled/plain")

dsname = "celeba32" if DATASET == "celeba" else DATASET
model_type = MODEL_TYPE
device = torch.device(
    ("cuda" if torch.cuda.is_available() else "cpu") if DEVICE == "auto" else DEVICE
)
ckpt_root = (
    Path(CKPT_ROOT).expanduser()
    if CKPT_ROOT is not None
    else TOPDIR / "model_checkpoints" / dsname
)
saved_dir = (
    Path(SAVED_TENSORS_DIR).expanduser()
    if SAVED_TENSORS_DIR is not None
    else TOPDIR / "saved_tensors" / dsname
)
seeds_path = (
    Path(SEEDS_FILE).expanduser() if SEEDS_FILE is not None else saved_dir / "seeds.pt"
)
reference_paths = {
    i: (
        Path(REFERENCE_FILES[i]).expanduser()
        if REFERENCE_FILES[i] is not None
        else saved_dir / f"subset{i}.pt"
    )
    for i in (1, 2)
}

plt.rcParams["font.family"] = FONT_FAMILY
plt.rcParams["font.serif"] = SERIF_FONTS


def checkpoint_dir(kind, subset):
    override = CHECKPOINT_DIR_OVERRIDES.get((kind, subset))
    if override is not None:
        return Path(override).expanduser()
    return ckpt_root / CHECKPOINT_NAME_TEMPLATE.format(
        kind=kind, subset=subset, suffix=MODEL_RUN_SUFFIXES[kind]
    )


def load_image_tensor(path):
    # Load trusted experiment tensors and checkpoints.
    value = torch.load(path, map_location="cpu", weights_only=False)
    if not isinstance(value, torch.Tensor) or value.ndim != 4:
        raise ValueError(f"{path}: expected a tensor of shape [N,C,H,W]")
    value = value.detach().float().cpu()
    if not torch.isfinite(value).all():
        raise ValueError(f"{path}: contains non-finite values")
    return value


seeds = load_image_tensor(seeds_path)
needed_subsets = set()
if RUN_TRAJECTORY or RUN_SINGLE_EPOCH:
    needed_subsets.add(ANALYSIS_SUBSET)
if RUN_SUBSET_PAIR or RUN_EARLY_LATE or RUN_CROSS_MODEL:
    needed_subsets.update((1, 2))
references = {i: load_image_tensor(reference_paths[i]) for i in sorted(needed_subsets)}
for i, reference in references.items():
    if reference.shape[1:] != seeds.shape[1:]:
        raise ValueError(
            f"Subset {i}: reference shape {reference.shape} does not match seeds {seeds.shape}"
        )
n_samples = min([NUM_SAMPLES, len(seeds)] + [len(x) for x in references.values()])
if n_samples < 1:
    raise ValueError("No aligned seed/reference samples are available")
seeds = seeds[:n_samples].contiguous()
references = {i: x[:n_samples].contiguous() for i, x in references.items()}
plot_indices = [int(i) for i in PLOT_INDICES if 0 <= i < n_samples]
if len(plot_indices) != len(PLOT_INDICES):
    print("Some PLOT_INDICES are outside the available samples and will be omitted.")
sample_cache = {}
sampling_records = {}
print(f"Dataset: {dsname}; device: {device}; paired samples: {n_samples}")
print("Noise:", seeds_path)
for kind in sorted({MODEL_TYPE} | ({"dit", "unet"} if RUN_CROSS_MODEL else set())):
    print(kind, ":", checkpoint_dir(kind, 1), "|", checkpoint_dir(kind, 2))


In [ ]:
from diffusion_project.models import ConvDiT, AttentionUNet
from diffusion_project.diffusion import DiffusionSchedule


def denormalize_img(x):
    return (x * 0.5 + 0.5).clamp(0, 1)


In [ ]:
def available_checkpoints(directory):
    return {
        int(path.stem[6:]): path
        for path in Path(directory).glob("epoch_*.pt")
        if path.stem[6:].isdigit()
    }


def resolve_epoch(directory, epoch):
    files = available_checkpoints(directory)
    if not files:
        raise FileNotFoundError(f"No epoch checkpoints in {directory}")
    selected = max(files) if epoch is None else int(epoch)
    if selected not in files:
        raise FileNotFoundError(
            f'Missing checkpoint: {Path(directory) / f"epoch_{selected:04d}.pt"}'
        )
    return selected, files


def architecture_kwargs(kind, config):
    keys = (
        (
            "image_size",
            "hidden_size",
            "patch_size",
            "depth",
            "num_heads",
            "mlp_ratio",
            "dropout",
        )
        if kind == "dit"
        else (
            "image_size",
            "model_channels",
            "channel_mult",
            "num_res_blocks",
            "attention_resolutions",
            "num_heads",
            "dropout",
        )
    )
    missing = [
        key for key in ("channels", "diffusion_steps") + keys if key not in config
    ]
    if missing:
        raise ValueError(f"{kind} checkpoint config is missing: {missing}")
    return dict(in_channels=config["channels"], **{key: config[key] for key in keys})


def load_averaged_model(kind, subset, epoch, avg_k=AVG_K):
    directory = checkpoint_dir(kind, subset)
    selected, files = resolve_epoch(directory, epoch)
    window = list(range(max(1, selected - avg_k + 1), selected + 1))
    used_epochs = [e for e in window if e in files]
    missing = sorted(set(window) - set(used_epochs))
    if missing:
        warnings.warn(
            f"{directory.name}: missing averaging epochs {missing}; using {used_epochs}"
        )
    avg_state = None
    config = None
    for e in used_epochs:
        checkpoint = torch.load(files[e], map_location="cpu", weights_only=False)
        current_config = checkpoint["config"]
        if int(checkpoint["epoch"]) != e:
            raise ValueError(f"{files[e]}: saved epoch disagrees with filename")
        kwargs = architecture_kwargs(kind, current_config)
        if config is not None and (
            kwargs != architecture_kwargs(kind, config)
            or current_config["diffusion_steps"] != config["diffusion_steps"]
        ):
            raise ValueError(
                f"{directory}: incompatible configs within averaging window"
            )
        data_config = checkpoint.get("data_config")
        if data_config is not None:
            saved_dataset = (
                "celeba32"
                if data_config["dataset"] == "celeba"
                else data_config["dataset"]
            )
            if saved_dataset != dsname or data_config["subset_size"] != SUBSET_SIZE:
                raise ValueError(
                    f"{files[e]}: dataset/subset size does not match the initial cell"
                )
        config = dict(current_config)
        state = checkpoint["model_state_dict"]
        if avg_state is None:
            avg_state = {key: value.clone() for key, value in state.items()}
        else:
            if state.keys() != avg_state.keys():
                raise ValueError(f"{directory}: checkpoint state keys differ")
            for key, value in state.items():
                if value.shape != avg_state[key].shape:
                    raise ValueError(
                        f"{directory}: checkpoint tensor shape differs for {key}"
                    )
                if torch.is_floating_point(value):
                    avg_state[key].add_(value)
                else:
                    avg_state[key] = value.clone()  # use latest non-floating buffer
        del state, checkpoint
    for value in avg_state.values():
        if torch.is_floating_point(value):
            value.div_(len(used_epochs))
    expected_shape = (config["channels"], config["image_size"], config["image_size"])
    if tuple(seeds.shape[1:]) != expected_shape:
        raise ValueError(
            f"{directory}: checkpoint expects {expected_shape}, seeds have {tuple(seeds.shape[1:])}"
        )
    model = (ConvDiT if kind == "dit" else AttentionUNet)(
        **architecture_kwargs(kind, config)
    )
    model.load_state_dict(avg_state, strict=True)
    model.to(device).eval()
    print(f"{kind}, subset {subset}, epoch {selected}: averaged {used_epochs}")
    return model, config, selected, used_epochs


@torch.no_grad()
def ddim_sample(model, fixed_noise, schedule, sample_steps=SAMPLE_STEPS):
    model.eval()
    if not 1 <= sample_steps <= schedule.T:
        raise ValueError(
            "SAMPLE_STEPS must be between 1 and the checkpoint diffusion_steps"
        )
    x = fixed_noise.to(device).clone()
    step_ids = torch.linspace(schedule.T - 1, 0, sample_steps, device=device).long()
    for i, t_scalar in enumerate(step_ids):
        t = torch.full(
            (x.shape[0],), int(t_scalar.item()), device=device, dtype=torch.long
        )
        v = model(x, t)
        x0, eps = schedule.predict_x0_eps_from_v(x, t, v)
        x0 = x0.clamp(-1.0, 1.0)
        if i == len(step_ids) - 1:
            x = x0
        else:
            t_prev = torch.full(
                (x.shape[0],),
                int(step_ids[i + 1].item()),
                device=device,
                dtype=torch.long,
            )
            alpha_prev = schedule.gather(schedule.sqrt_alphas_cumprod, t_prev, x.shape)
            sigma_prev = schedule.gather(
                schedule.sqrt_one_minus_alphas_cumprod, t_prev, x.shape
            )
            x = alpha_prev * x0 + sigma_prev * eps
    return x.cpu()


def get_samples(kind, subset, epoch):
    selected, _ = resolve_epoch(checkpoint_dir(kind, subset), epoch)
    key = (kind, subset, selected)
    if key in sample_cache:
        return sample_cache[key]
    model, config, selected, used_epochs = load_averaged_model(kind, subset, selected)
    schedule = DiffusionSchedule(config["diffusion_steps"], device)
    try:
        samples = torch.cat(
            [
                ddim_sample(model, chunk, schedule)
                for chunk in seeds.split(SAMPLING_BATCH_SIZE)
            ],
            dim=0,
        )
    finally:
        del model, schedule
    if not torch.isfinite(samples).all():
        raise ValueError(
            f"{kind}, subset {subset}, epoch {selected}: non-finite samples"
        )
    sampling_records[key] = {
        "checkpoint_dir": str(checkpoint_dir(kind, subset)),
        "epoch": selected,
        "averaged_epochs": used_epochs,
        "config": config,
    }
    if CACHE_SAMPLES:
        sample_cache[key] = samples
    return samples


In [ ]:
def as_numpy(value):
    return value.detach().cpu().numpy() if torch.is_tensor(value) else np.asarray(value)


def per_image_corr(a, b):
    a, b = as_numpy(a), as_numpy(b)
    if a.shape != b.shape or a.ndim != 4:
        raise ValueError("Correlation expects matching [N,C,H,W] arrays")
    a = a.reshape(len(a), -1).astype(np.float64)
    b = b.reshape(len(b), -1).astype(np.float64)
    a -= a.mean(axis=1, keepdims=True)
    b -= b.mean(axis=1, keepdims=True)
    numerator = np.sum(a * b, axis=1)
    denominator = np.sqrt(np.sum(a * a, axis=1) * np.sum(b * b, axis=1))
    result = np.full(len(a), np.nan)
    np.divide(numerator, denominator, out=result, where=denominator > 0)
    return np.clip(result, -1.0, 1.0)


def median_corr(values):
    valid = np.asarray(values)[np.isfinite(values)]
    return float(np.median(valid)) if len(valid) else float("nan")


def image_for_display(value):
    value = as_numpy(denormalize_img(value))
    return value[0] if value.shape[0] == 1 else value.transpose(1, 2, 0)


def show_image(value, title=None, axes_off=False):
    fig, ax = plt.subplots(figsize=INDIVIDUAL_FIGSIZE)
    ax.imshow(image_for_display(value), cmap=IMAGE_CMAP)
    if title is not None:
        ax.set_title(title)
    if axes_off:
        ax.axis("off")
    if SAVE_FIGURES:
        FIGURE_DIR.mkdir(parents=True, exist_ok=True)
        index = len(list(FIGURE_DIR.glob("comparison_*.pdf")))
        plt.gcf().savefig(
            FIGURE_DIR / f"comparison_{index:04d}.pdf", bbox_inches="tight"
        )
    plt.show()
    plt.close(fig)


def show_error(value, title=None, vmin=None, vmax=None, signed=False):
    fig, ax = plt.subplots(figsize=INDIVIDUAL_FIGSIZE)
    ax.imshow(
        value, vmin=vmin, vmax=vmax, cmap=SIGNED_ERROR_CMAP if signed else ERROR_CMAP
    )
    if title is not None:
        ax.set_title(title)
    if SAVE_FIGURES:
        FIGURE_DIR.mkdir(parents=True, exist_ok=True)
        index = len(list(FIGURE_DIR.glob("comparison_*.pdf")))
        plt.gcf().savefig(
            FIGURE_DIR / f"comparison_{index:04d}.pdf", bbox_inches="tight"
        )
    plt.show()
    plt.close(fig)


def show_grid(rows, figsize, titles=None, rect=None):
    fig, axes = plt.subplots(len(rows), len(rows[0]), figsize=figsize, squeeze=False)
    for r, row in enumerate(rows):
        for c, image in enumerate(row):
            axes[r, c].imshow(image_for_display(image), cmap=IMAGE_CMAP)
            axes[r, c].axis("off")
    if titles is not None:
        for c, title in enumerate(titles):
            axes[0, c].set_title(title, fontsize=GRID_TITLE_SIZE, pad=GRID_TITLE_PAD)
    if rect is None:
        plt.tight_layout()
    else:
        plt.tight_layout(rect=rect)
    if SAVE_FIGURES:
        FIGURE_DIR.mkdir(parents=True, exist_ok=True)
        index = len(list(FIGURE_DIR.glob("comparison_*.pdf")))
        plt.gcf().savefig(
            FIGURE_DIR / f"comparison_{index:04d}.pdf", bbox_inches="tight"
        )
    plt.show()
    plt.close(fig)


In [ ]:
# Optional compact trajectory; full trajectories are handled by notebook 03.
if RUN_TRAJECTORY:
    rows = []
    files = available_checkpoints(checkpoint_dir(MODEL_TYPE, ANALYSIS_SUBSET))
    for epoch in TRAJECTORY_EPOCHS:
        if epoch in files:
            samples = get_samples(MODEL_TYPE, ANALYSIS_SUBSET, epoch)
            reference = references[ANALYSIS_SUBSET]
            rows.append(
                (
                    epoch,
                    median_corr(per_image_corr(samples, reference)),
                    float((samples - reference).square().mean()),
                )
            )
    if rows:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        axes[0].plot([r[0] for r in rows], [r[1] for r in rows])
        axes[0].set_ylabel("Median corr.")
        axes[1].plot([r[0] for r in rows], [r[2] for r in rows])
        axes[1].set_ylabel("MSE")
        for ax in axes:
            ax.set_xlabel("Epoch")
        plt.show()


In [ ]:
if RUN_SINGLE_EPOCH:
    single_samples = get_samples(MODEL_TYPE, ANALYSIS_SUBSET, SINGLE_EPOCH)
    single_reference = references[ANALYSIS_SUBSET]
    single_corr = per_image_corr(single_samples, single_reference)
    for i in plot_indices:
        print(f"NUMBER {i}")
        show_image(single_reference[i], f"corr. = {single_corr[i]:.4f}", axes_off=True)
        show_image(single_samples[i], axes_off=True)


In [ ]:
if RUN_SUBSET_PAIR:
    pair_samples = {i: get_samples(MODEL_TYPE, i, PAIR_EPOCHS[i]) for i in (1, 2)}
    for i in plot_indices:
        print(f"NUMBER {i}")
        show_grid(
            [
                [pair_samples[1][i], references[1][i]],
                [pair_samples[2][i], references[2][i]],
            ],
            PAIR_FIGSIZE,
        )


In [ ]:
if RUN_EARLY_LATE:
    early_samples = {
        i: get_samples(MODEL_TYPE, i, EARLY_LATE_EPOCHS[0]) for i in (1, 2)
    }
    late_samples = {i: get_samples(MODEL_TYPE, i, EARLY_LATE_EPOCHS[1]) for i in (1, 2)}
    for i in plot_indices:
        show_grid(
            [
                [references[1][i], early_samples[1][i], late_samples[1][i]],
                [references[2][i], early_samples[2][i], late_samples[2][i]],
            ],
            EARLY_LATE_FIGSIZE,
        )


In [ ]:
if RUN_CROSS_MODEL:
    cross_samples = {
        (kind, subset): get_samples(kind, subset, CROSS_MODEL_EPOCHS[kind])
        for kind in ("unet", "dit")
        for subset in (1, 2)
    }
    for i in plot_indices:
        rows = [
            [
                references[subset][i],
                cross_samples["unet", subset][i],
                cross_samples["dit", subset][i],
            ]
            for subset in (1, 2)
        ]
        for variant in CROSS_GRID_VARIANTS:
            if variant == "titled":
                show_grid(
                    rows,
                    CROSS_FIGSIZE,
                    titles=(REFERENCE_LABEL, "UNet", "DiT"),
                    rect=TITLED_CROSS_RECT,
                )
            else:
                titles = ("Analytical", "UNet", "DiT") if PLAIN_CROSS_TITLES else None
                show_grid(rows, CROSS_FIGSIZE, titles=titles)
